In [1]:
# ¿Cómo transformamos los abstracts de publicaciones de Scopus en una matriz documento–término que permita analizar su contenido?

import pandas as pd

dataframe = pd.read_csv("../data/scopus_abstracts.csv.gz", compression="gzip")
dataframe[["scopus_record_id", "publication_title", "abstract"]].head()

,scopus_record_id,publication_title,abstract
0,2-s2.0-105042759102,Strategic Pathways in Malaysia’s Fintech Secto...,This study examines the relationship between e...
1,2-s2.0-85175999008,"ELI Principles on Blockchain Technology, Smart...",The project started in 2018 following a Use-Ca...
2,2-s2.0-85149119289,Bitland-A Decentralized Commercial Real Estate...,Real estate has always been a significant inve...
3,2-s2.0-84931062324,"The politics of person, property, technology: ...",Global policy organizations have emphasized ca...
4,2-s2.0-85049126196,Planned preventative maintenance and proptech,[No abstract available]


In [2]:
# La normalización conserva el contenido de cada abstract y elimina diferencias superficiales de escritura.

dataframe["normalized_text"] = dataframe["abstract"].str.lower().str.split().str.join(" ")
dataframe[["abstract", "normalized_text"]].head()

,abstract,normalized_text
0,This study examines the relationship between e...,this study examines the relationship between e...
1,The project started in 2018 following a Use-Ca...,the project started in 2018 following a use-ca...
2,Real estate has always been a significant inve...,real estate has always been a significant inve...
3,Global policy organizations have emphasized ca...,global policy organizations have emphasized ca...
4,[No abstract available],[no abstract available]


In [3]:
# Se separa cada documento en tokens para que el texto se convierta en unidades procesables.

from nltk.tokenize import word_tokenize

dataframe["tokens"] = dataframe["normalized_text"].map(word_tokenize)
dataframe[["normalized_text", "tokens"]].head()

,normalized_text,tokens
0,this study examines the relationship between e...,"[this, study, examines, the, relationship, bet..."
1,the project started in 2018 following a use-ca...,"[the, project, started, in, 2018, following, a..."
2,real estate has always been a significant inve...,"[real, estate, has, always, been, a, significa..."
3,global policy organizations have emphasized ca...,"[global, policy, organizations, have, emphasiz..."
4,[no abstract available],"[[, no, abstract, available, ]]"


In [4]:
# Se conservan tokens alfabéticos y se hace visible la información que la regla descarta.

dataframe["tokens"] = dataframe["tokens"].map(
    lambda tokens: [token for token in tokens if token.isalpha()]
)
dataframe["processed_text"] = dataframe["tokens"].map(" ".join)
dataframe[["normalized_text", "processed_text"]].head()

,normalized_text,processed_text
0,this study examines the relationship between e...,this study examines the relationship between e...
1,the project started in 2018 following a use-ca...,the project started in following a approach ie...
2,real estate has always been a significant inve...,real estate has always been a significant inve...
3,global policy organizations have emphasized ca...,global policy organizations have emphasized pr...
4,[no abstract available],no abstract available


In [5]:
# ¿Cómo se transforma la lista de tokens en una matriz documento-término apta para clasificación?

from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(min_df=2)
document_term_matrix = vectorizer.fit_transform(dataframe["processed_text"])
vocabulary = pd.DataFrame(
    {"token": vectorizer.get_feature_names_out(), "document_frequency": (document_term_matrix > 0).sum(axis=0).A1}
)
document_term_matrix.shape, vocabulary.head()

((405, 3952),
           token  document_frequency
 0  abbreviation                   2
 1       ability                  15
 2          able                   9
 3         about                  48
 4         above                   6)

In [6]:
# Se guardan abstracts procesados, vocabulario y matriz para conservar una representación reproducible del corpus.

import json
from pathlib import Path
from scipy import sparse

submission_dir = Path("../submission")
dataframe.drop(columns="tokens").to_csv(submission_dir / "tokenized_abstracts.csv", index=False)
vocabulary.to_csv(submission_dir / "vocabulary.csv", index=False)
sparse.save_npz(submission_dir / "document_term_matrix.npz", document_term_matrix)
with (submission_dir / "matrix_metadata.json").open("w", encoding="utf-8") as file:
    json.dump(
        {
            "documents": document_term_matrix.shape[0],
            "terms": document_term_matrix.shape[1],
            "minimum_document_frequency": 2,
        },
        file,
        indent=2,
    )